# Aprendizagem de Máquina — Web Academy
## Prática de Agrupamento (Clustering) — Segmentação de Clientes com RFM + K-Means

Pós-graduação em Desenvolvimento Web — Instituto de Computação / UFAM

## 1.1. O Problema de Negócio: Por Que Segmentar Clientes?

**A premissa central:**

- No e-commerce vale uma verdade simples mas poderosa: nem todos os clientes são iguais, e tratá-los como se fossem desperdiça recursos.
- Campanhas de marketing "tamanho único" tendem a ser ineficientes e caras, porque falam da mesma forma com quem acabou de comprar e com quem sumiu há um ano.

**O que a segmentação resolve:**

- Ela divide a base de consumidores em grupos homogêneos, reunindo pessoas com comportamento e características de compra parecidos.
- Com esses grupos em mãos, a empresa consegue personalizar campanhas, otimizar o retorno sobre o investimento (ROI) em marketing e, sobretudo, melhorar a retenção dos clientes mais valiosos.

**O objetivo desta aula:**

- Vamos transformar um arquivo com mais de um milhão de registros transacionais em um conjunto pequeno e acionável de segmentos de clientes.
- A meta é chegar a 3–5 grupos bem definidos — como "Clientes Campeões", "Clientes em Risco" e "Novos Clientes" — que a equipe de marketing possa usar imediatamente para criar campanhas direcionadas.
- No fim, o que era um amontoado de dados brutos vira inteligência de negócio pronta para ação.

## 1.2. O Modelo RFM: A Lógica por Trás dos Dados

**O que é RFM:**

- RFM é um acrônimo para **Recência, Frequência e Valor Monetário**, três dimensões que resumem o relacionamento de cada cliente com a loja.
- Mais do que um punhado de métricas estatísticas, é um modelo de comportamento consagrado no marketing, que quantifica o valor e o engajamento de cada consumidor a partir do seu histórico de transações.

**As três métricas, uma a uma:**

- **Recência (R)** responde "quando foi a última vez que o cliente comprou?". A lógica é que quem comprou há pouco tem a marca fresca na memória e é mais propenso a responder a novas ofertas — por isso uma recência baixa (poucos dias desde a última compra) é um sinal positivo.
- **Frequência (F)** responde "com que frequência esse cliente compra?". Compras repetidas demonstram lealdade e engajamento, de modo que uma frequência alta costuma indicar um cliente satisfeito e retido.
- **Valor Monetário (M)** responde "quanto o cliente gasta no total?". Essa métrica identifica quem mais contribui para a receita, e clientes com valor monetário alto são, por definição, extremamente valiosos para o negócio.

**A base conceitual:**

- O modelo parte da premissa de que o comportamento de compra passado é o melhor preditor do comportamento futuro.
- Ele se conecta diretamente ao **Princípio de Pareto (a regra 80/20)**, segundo o qual cerca de 80% da receita costuma vir de aproximadamente 20% dos clientes.
- Assim, o RFM oferece um framework quantitativo tanto para identificar esses 20% vitais quanto para diagnosticar grupos em risco de abandono (churn), o que faz dele uma verdadeira ferramenta estratégica de diagnóstico, e não apenas um exercício de análise de dados.

## 1.3. Nosso Plano de Ação

Vamos seguir um roteiro estruturado, que reflete a sequência de um projeto de ciência de dados do mundo real:

1. **Dados brutos** — começamos carregando o dataset transacional de e-commerce, no estado em que ele chega.
2. **Limpeza e preparação** — enfrentamos os problemas típicos de dados reais, como valores ausentes, cancelamentos e registros inválidos.
3. **Engenharia de features** — a etapa crucial em que calculamos Recência, Frequência e Valor Monetário para cada cliente.
4. **Pré-processamento** — tratamos a forte assimetria das métricas com log e depois padronizamos a escala, dois passos essenciais para o modelo funcionar bem.
5. **Modelagem** — aplicamos o K-Means para agrupar os clientes e comparamos o resultado com o clustering hierárquico.
6. **Interpretação** — o passo mais importante, no qual traduzimos clusters matemáticos em personas de clientes compreensíveis e acionáveis.
7. **Desafio final** — consolidamos tudo aplicando o processo completo a um dataset novo e mais complexo (Olist).

**O que esta versão acrescenta ao roteiro clássico:**

- Uma **transformação logarítmica** para lidar com a cauda longa de Frequência e Valor Monetário, que sem tratamento arruína o K-Means.
- O **Silhouette Score** ao lado do Método do Cotovelo, dando um critério numérico para escolher o número de clusters.
- Uma **nomeação de clusters derivada dos centróides**, robusta a mudanças de execução, no lugar de um mapa fixo que quebra facilmente.
- Uma comparação com o **clustering hierárquico** (dendrograma) e uma seção didática de **RFM por quantis** (`qcut`), o scoring clássico de marketing sem machine learning.
- Visualizações extras (clusters em 2D) e um **perfil de negócio** com percentual da base e da receita por segmento.

## Seção 2: Preparando o Terreno — Carregamento e Limpeza dos Dados

### 2.1. O Dataset "Online Retail II"

**Origem e conteúdo:**

- O ponto de partida é o dataset **Online Retail II**, disponibilizado pelo UCI Machine Learning Repository, que reúne transações reais de um varejista online do Reino Unido.
- Ele cobre dois anos de operação, de dezembro de 2009 a dezembro de 2011, o que dá material suficiente para medir recência e frequência com sentido.
- A empresa vende principalmente artigos para presente, e boa parte de seus clientes são atacadistas — o que explica a presença de transações com quantidades muito grandes e será importante quando falarmos de outliers.

**Dicionário de dados:**

- `Invoice` é o número da fatura; quando começa com a letra **'C'**, indica um cancelamento.
- `StockCode` e `Description` identificam, respectivamente, o código e o nome do produto.
- `Quantity` traz a quantidade de itens da transação e pode vir negativa em casos de devolução.
- `InvoiceDate` guarda a data e a hora da compra, base para o cálculo da recência.
- `Price` é o preço unitário em libras esterlinas (£), e `Customer ID` é o identificador do cliente, peça essencial para toda a análise.
- `Country` informa o país de residência do cliente.

In [ ]:
# Bibliotecas
import pandas as pd
import numpy as np
import datetime as dt
import matplotlib.pyplot as plt
import seaborn as sns

# Configurações de exibição
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 500)
pd.set_option('display.float_format', lambda x: '%.3f' % x)

# O dataset vem em duas abas (2009-2010 e 2010-2011). Carregamos e concatenamos.
# Download: https://archive.ics.uci.edu/dataset/502/online+retail+ii
try:
    df_2009_2010 = pd.read_excel('online_retail_II.xlsx', sheet_name='Year 2009-2010')
    df_2010_2011 = pd.read_excel('online_retail_II.xlsx', sheet_name='Year 2010-2011')
    df = pd.concat([df_2009_2010, df_2010_2011], ignore_index=True)
except FileNotFoundError:
    print("Arquivo 'online_retail_II.xlsx' não encontrado. Usando um dataframe de exemplo.")
    data_exemplo = {
        'Invoice': ['536365', '536365', '536366', 'C536379', '536381', '536381'],
        'StockCode': ['85123A', '71053', '22633', 'D', '22423', '85099B'],
        'Description': ['CREAM CUPID HEARTS COAT HANGER', 'WHITE METAL LANTERN',
                        'HAND WARMER UNION JACK', 'Discount',
                        'REGENCY CAKESTAND 3 TIER', 'JUMBO BAG RED RETROSPOT'],
        'Quantity': [6, 6, 2, -1, 8, 2],
        'InvoiceDate': [dt.datetime(2010, 12, 1, 8, 26), dt.datetime(2010, 12, 1, 8, 26),
                        dt.datetime(2010, 12, 1, 8, 28), dt.datetime(2010, 12, 1, 9, 41),
                        dt.datetime(2010, 12, 1, 9, 41), dt.datetime(2010, 12, 1, 9, 41)],
        'Price': [2.75, 3.39, 1.85, 27.50, 12.75, 1.95],
        'Customer ID': [17850.0, 17850.0, 17850.0, 14527.0, 15311.0, 15311.0],
        'Country': ['United Kingdom'] * 6,
    }
    df = pd.DataFrame(data_exemplo)

print("Primeiras 5 linhas:")
print(df.head())
print("\nInformações gerais:")
df.info()
print("\nEstatísticas descritivas:")
print(df.describe().T)

### 2.2. A Realidade dos Dados Brutos: Limpeza como EDA

**Por que a limpeza já é análise, e não só preparação:**

- É comum tratar a limpeza como uma etapa preliminar chata, mas na prática ela é a primeira e mais crítica fase da Análise Exploratória de Dados (EDA).
- Cada decisão de limpeza reflete uma regra de negócio: como nosso objetivo é medir o valor de um cliente, transações canceladas, produtos devolvidos e compras de clientes não identificados simplesmente não agregam valor e não podem entrar na conta.
- Em outras palavras, a limpeza é o que define o escopo daquilo que consideramos um "cliente analisável".

**Os passos que vamos aplicar:**

- Remover registros **sem `Customer ID`**, já que sem a chave do cliente não há como atribuir a compra a ninguém.
- Remover **cancelamentos**, identificados pelas faturas que começam com 'C'.
- Filtrar **quantidades não positivas** (devoluções) e **preços iguais ou menores que zero**, que não representam compras válidas.
- Remover **duplicatas**, para não contar a mesma transação duas vezes.

> **Correção técnica importante:** aplicamos `.copy()` logo após o primeiro filtro. Sem isso, as atribuições seguintes (como `df_cleaned['Invoice'] = ...`) disparam o `SettingWithCopyWarning` do pandas e podem não persistir de forma confiável, um erro silencioso clássico.

In [ ]:
print(f"Formato original: {df.shape}")

# 1. Valores ausentes
print("\nNulos por coluna:")
print(df.isnull().sum())

# Customer ID é a chave -> .copy() evita SettingWithCopyWarning nas edições seguintes
df_cleaned = df.dropna(subset=['Customer ID']).copy()
print(f"\nApós remover nulos em Customer ID: {df_cleaned.shape}")

# 2. Cancelamentos (Invoice começa com 'C')
df_cleaned['Invoice'] = df_cleaned['Invoice'].astype(str)
df_cleaned = df_cleaned[~df_cleaned['Invoice'].str.startswith('C')]
print(f"Após remover cancelamentos: {df_cleaned.shape}")

# 3. Quantidades e preços inválidos
df_cleaned = df_cleaned[(df_cleaned['Quantity'] > 0) & (df_cleaned['Price'] > 0)]
print(f"Após remover quantidades/preços inválidos: {df_cleaned.shape}")

# 4. Duplicatas
print(f"\nLinhas duplicadas: {df_cleaned.duplicated().sum()}")
df_cleaned = df_cleaned.drop_duplicates()
print(f"Formato final após limpeza: {df_cleaned.shape}")

> **Nota sobre `startswith('C')` vs `contains('C')`:** o notebook original usava `str.contains('C')`, que casaria qualquer código de fatura com um 'C' em qualquer posição. Como o marcador de cancelamento é sempre o **prefixo** 'C', `str.startswith('C')` é mais preciso e evita descartar faturas legítimas por engano.

### Exercício Prático 1: Inspetor de Dados

**Tarefa:** limpe o dataframe de amostra abaixo aplicando as mesmas regras da aula — remover nulos em `CustomerID`, remover cancelamentos, remover quantidades negativas e remover duplicatas. Imprima o `.shape` antes e depois para ver quantas linhas cada regra elimina.

> Repare que aqui a coluna se chama `CustomerID` (sem espaço), diferente do dataset real, em que é `Customer ID` (com espaço). Em dados do mundo real os nomes de coluna variam bastante, então confira sempre o nome exato antes de escrever o código — esse é um erro muito comum na prática.

In [ ]:
data_exercicio = {
    'Invoice': ['536365', 'C536368', '536369', '536370', '536370', '536371'],
    'Quantity': [6, -1, 3, 20, 20, 8],
    'Price': [2.55, 4.25, 3.75, 7.85, 7.85, 2.08],
    'CustomerID': [17850.0, 13047.0, 13047.0, 12583.0, 12583.0, np.nan]
}
df_exercicio = pd.DataFrame(data_exercicio)
print(f"Formato inicial: {df_exercicio.shape}")

# --- SEU CÓDIGO DE LIMPEZA AQUI ---
# 1. Remover nulos em CustomerID
# 2. Remover cancelamentos (dica: str.startswith('C'))
# 3. Remover quantidades negativas
# 4. Remover duplicatas

# print(f"Formato final: {df_exercicio_limpo.shape}")
# print(df_exercicio_limpo)

## Seção 3: Engenharia de Features — Construindo as Métricas RFM

**A mudança de granularidade que precisamos fazer:**

- Neste momento os dados estão no grão de **transação**: cada linha é um item comprado em um instante específico.
- Para segmentar clientes, precisamos de um grão diferente — **uma linha por cliente**, resumindo todo o histórico dele em três números (R, F e M).
- Essa passagem de "transações" para "perfis de cliente" é exatamente o que chamamos de engenharia de features neste projeto.

### 3.1. A Base do Cálculo: a Coluna `TotalPrice`

**Por que ela precisa ser criada:**

- O valor gasto em cada transação não vem pronto no dataset; temos apenas o preço unitário e a quantidade separadamente.
- Calculamos então `TotalPrice = Quantity × Price`, que representa o valor total de cada linha de compra.
- Essa coluna será a matéria-prima da métrica Monetary (M), que soma todos os `TotalPrice` de um mesmo cliente.

> Como `df_cleaned` já veio de um `.copy()` na seção anterior, esta atribuição é segura e não dispara o aviso de cópia do pandas.

In [ ]:
df_cleaned['TotalPrice'] = df_cleaned['Quantity'] * df_cleaned['Price']
print("Coluna 'TotalPrice' criada:")
print(df_cleaned[['Quantity', 'Price', 'TotalPrice']].head())

### 3.2. Calculando R, F e M e Consolidando a Tabela RFM

**O ponto de referência temporal (snapshot):**

- A recência mede "há quantos dias foi a última compra", então precisamos de uma data que funcione como o "hoje" da análise.
- A prática comum é pegar a data da transação mais recente de todo o dataset e somar um dia, criando um "hoje" hipotético logo após o fim dos dados.
- A partir desse ponto de referência, medimos para cada cliente quantos dias se passaram desde a sua última compra.

**As três agregações, em uma única passada com `.agg()`:**

- A **Recência** sai da diferença, em dias, entre o snapshot e a data da última compra de cada cliente.
- A **Frequência** vem da contagem de faturas **únicas** (`nunique`), porque o que interessa é quantas vezes o cliente comprou, não quantos itens levou.
- O **Monetary** é a **soma** de `TotalPrice`, ou seja, o total gasto pelo cliente ao longo de todo o período.

In [ ]:
df_cleaned['InvoiceDate'] = pd.to_datetime(df_cleaned['InvoiceDate'])

snapshot_date = df_cleaned['InvoiceDate'].max() + dt.timedelta(days=1)
print(f"Data de referência (snapshot): {snapshot_date}")

rfm = df_cleaned.groupby('Customer ID').agg(
    Recency=('InvoiceDate', lambda date: (snapshot_date - date.max()).days),
    Frequency=('Invoice', 'nunique'),
    Monetary=('TotalPrice', 'sum'),
)

print("\nTabela RFM:")
print(rfm.head())
print("\nEstatísticas:")
print(rfm.describe().T)

> Esta tabela é a materialização do **vetor de características** que descreve cada cliente. É a partir dela — e não mais das transações individuais — que o algoritmo de clusterização vai aprender os padrões de comportamento.

### Exercício Prático 2: Mestre da Frequência

**Tarefa:** usando o dataframe limpo (`df_cleaned`), calcule a Frequência dos **5 clientes com os maiores `Customer ID`** e descubra qual deles realizou o maior número de compras.

**Dicas para chegar lá:**

- Primeiro, obtenha os cinco maiores identificadores, por exemplo com `sorted(df_cleaned['Customer ID'].unique())[-5:]`.
- Em seguida, filtre o dataframe para manter apenas as transações desses clientes.
- Por fim, agrupe por cliente e conte as faturas únicas com `nunique`, que é justamente a definição de frequência usada na aula.

In [ ]:
# --- SEU CÓDIGO AQUI ---
# maiores_ids = sorted(df_cleaned['Customer ID'].unique())[-5:]
# sub = df_cleaned[df_cleaned['Customer ID'].isin(maiores_ids)]
# print(sub.groupby('Customer ID')['Invoice'].nunique())

## Seção 4: Distribuição, Outliers e Escala — Preparando os Dados para o K-Means

Esta é a seção em que mais nos afastamos do notebook original, porque é aqui que mora a diferença entre clusters úteis e clusters degenerados. O K-Means é muito sensível não apenas à escala das features, mas também ao **formato** da distribuição de cada uma.

### 4.1. Diagnóstico: a Distribuição RFM é Assimétrica

**O que costuma acontecer com dados RFM:**

- A Recência tende a ser relativamente comportada, sem valores absurdamente extremos.
- Já a Frequência e o Monetary quase sempre têm **cauda longa à direita**: a grande maioria dos clientes compra pouco e gasta pouco, enquanto uns poucos atacadistas compram e gastam quantias muito acima do resto.

**Por que essa assimetria atrapalha o K-Means:**

- O K-Means assume que os clusters têm forma aproximadamente esférica e mede tudo por distância euclidiana.
- Com uma cauda longa, poucos pontos extremos passam a dominar completamente o cálculo de distância, "puxando" os centróides na direção deles.
- O resultado típico é péssimo: um cluster gigante engolindo quase todo mundo e outros clusters minúsculos que só capturam alguns outliers — uma segmentação inútil para o negócio.

Por isso, antes de decidir o que fazer, vamos **visualizar** essa assimetria com histogramas e medir o coeficiente de assimetria (skew).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, ['Recency', 'Frequency', 'Monetary']):
    sns.histplot(rfm[col], bins=50, ax=ax)
    ax.set_title(f'{col} (assimetria = {rfm[col].skew():.2f})')
plt.tight_layout()
plt.show()

print("Assimetria (skew) — quanto maior, mais cauda à direita:")
print(rfm[['Recency', 'Frequency', 'Monetary']].skew())

### 4.2. Discutindo Outliers: Remover, Capar ou Transformar?

**As três abordagens possíveis, com seus prós e contras:**

- **Remover** os extremos (por exemplo, tudo acima do percentil 99) é simples, mas descarta clientes reais — e no varejo os atacadistas de altíssimo valor podem ser justamente os mais importantes de todos, então jogá-los fora costuma ser um tiro no pé.
- **Capar** (a técnica de *winsorize*) mantém o cliente na base mas limita o valor a um teto máximo, reduzindo a influência dos extremos; a desvantagem é que a escolha desse teto é sempre um tanto arbitrária.
- **Transformar** com logaritmo comprime a cauda longa sem descartar ninguém e sem inventar um teto, aproximando a distribuição de um formato mais simétrico, que é justamente o que o K-Means "gosta" de receber.

**Nossa decisão e o porquê dela:**

- Vamos aplicar a transformação **`log1p`** (logaritmo de 1+x, seguro mesmo quando o valor é zero) às métricas, comprimindo a cauda sem perder informação sobre a ordem dos clientes.
- Optamos por **não remover** os atacadistas, porque eles são clientes legítimos e valiosos que merecem estar na segmentação.
- Ainda assim, é boa prática **inspecionar** os maiores valores antes de seguir, e é isso que a célula abaixo faz ao listar os clientes de maior Monetary.

In [ ]:
# Inspeção rápida dos maiores Monetary (provavelmente atacadistas)
print("Top 5 clientes por Monetary:")
print(rfm.sort_values('Monetary', ascending=False).head())

# Quantos clientes acima do percentil 99 de Monetary?
p99 = rfm['Monetary'].quantile(0.99)
print(f"\nPercentil 99 de Monetary: {p99:.2f}")
print(f"Clientes acima do p99: {(rfm['Monetary'] > p99).sum()}")

### 4.3. Transformação Logarítmica (`log1p`)

**O que a transformação faz, na prática:**

- `log1p(x)` calcula `log(1 + x)`, o que garante que o valor 0 seja tratado sem erro (algo que o log puro não permitiria).
- Ao aplicar o log, os valores gigantescos da cauda são puxados para perto do corpo da distribuição, enquanto as diferenças entre os valores pequenos são preservadas.
- Depois desse passo, a padronização que faremos em seguida se torna muito mais eficaz, porque já não há mais um punhado de pontos distorcendo tudo.

**Como aplicamos:**

- Transformamos R, F e M e guardamos o resultado em um novo DataFrame chamado `rfm_log`, mantendo o `rfm` original intacto — vamos precisar dos valores originais lá na frente, na hora de interpretar os segmentos em reais e em dias.

In [ ]:
rfm_log = rfm[['Recency', 'Frequency', 'Monetary']].apply(np.log1p)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, ['Recency', 'Frequency', 'Monetary']):
    sns.histplot(rfm_log[col], bins=50, ax=ax, color='seagreen')
    ax.set_title(f'log1p({col}) — skew = {rfm_log[col].skew():.2f}')
plt.tight_layout()
plt.show()

print("Assimetria após log1p (compare com a Seção 4.1):")
print(rfm_log.skew())

### 4.4. Por que Padronizar Depois do Log?

**O problema da escala no cálculo de distância:**

- No K-Means, a distância entre dois clientes depende diretamente da escala de cada feature, então a métrica de maior magnitude acaba dominando o cálculo.
- Mesmo depois do log, R, F e M continuam em escalas diferentes entre si, o que ainda enviesaria o agrupamento.

**O papel do `StandardScaler`:**

- Ele transforma cada feature para ter **média 0 e desvio padrão 1**, aplicando o z-score $z = \dfrac{x - \mu}{\sigma}$.
- Com todas as métricas na mesma escala, cada uma passa a contribuir de forma equitativa para a distância, sem que Monetary "atropele" Recência só por ter números maiores.

**A ordem correta do pipeline** é, portanto: primeiro `log1p`, depois `StandardScaler`, e só então o K-Means.

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm_log)   # note: escalamos os dados JÁ transformados por log

rfm_scaled_df = pd.DataFrame(rfm_scaled, columns=rfm_log.columns, index=rfm_log.index)
print("Estatísticas após log + padronização (média ~0, desvio ~1):")
print(rfm_scaled_df.describe().T)

### 4.5. Visualizando o Antes e o Depois

- Os boxplots abaixo comparam a distribuição original de cada métrica com a versão já transformada por log e padronizada.
- Observe como, no gráfico da esquerda, os outliers extremos esmagam a caixa toda contra a base, tornando a distribuição ilegível.
- No gráfico da direita, depois do tratamento, as caixas ficam bem formadas e comparáveis entre si, que é exatamente a condição de que o K-Means precisa para funcionar bem.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
sns.boxplot(data=rfm[['Recency', 'Frequency', 'Monetary']], ax=axes[0])
axes[0].set_title('RFM — Original (com outliers dominando)')
sns.boxplot(data=rfm_scaled_df, ax=axes[1])
axes[1].set_title('RFM — Após log + padronização')
plt.tight_layout()
plt.show()

### Exercício Prático 3: Normalização Manual

**Tarefa:** dado o array `dados = np.array([[10],[20],[30],[40],[50]])`, calcule a média e o desvio padrão e aplique a fórmula do z-score manualmente a cada elemento, reproduzindo à mão o que o `StandardScaler` faz internamente.

**Extensão sugerida:** aplique também `np.log1p` aos dados **antes** de padronizar e compare os dois resultados. Como a transformação logarítmica muda os valores padronizados, e o que isso indicaria se os dados tivessem uma cauda longa de verdade?

In [ ]:
# --- SEU CÓDIGO AQUI ---
dados = np.array([[10], [20], [30], [40], [50]])
# 1. média e desvio padrão
# 2. z = (x - média) / desvio
# 3. (extensão) repita aplicando np.log1p(dados) antes

## Seção 5: Encontrando os Grupos — Clusterização com K-Means

### 5.1. Como o K-Means Funciona?

**O algoritmo, passo a passo:**

1. **Escolha de `k`** — antes de tudo, definimos quantos clusters queremos encontrar; esse é o único parâmetro obrigatório do método.
2. **Inicialização** — o algoritmo posiciona `k` **centróides** (os centros dos clusters); com a estratégia `k-means++`, esse chute inicial é feito de forma inteligente, e não puramente ao acaso, o que acelera a convergência.
3. **Atribuição** — cada cliente é atribuído ao cluster cujo centróide estiver mais próximo dele.
4. **Atualização** — cada centróide é recalculado como a média de todos os pontos que foram atribuídos a ele naquela rodada.
5. **Repetição** — os passos de atribuição e atualização se repetem até que os centróides parem de se mover de forma significativa, o que chamamos de convergência.

**O que o algoritmo tenta minimizar:**

- O objetivo interno é reduzir a **Inércia** (WCSS, ou *Within-Cluster Sum of Squares*), que é a soma das distâncias ao quadrado de cada ponto até o seu próprio centróide.
- Quanto menor a inércia, mais "compactos" são os clusters — mas, como veremos, inércia baixa não é bom por si só.

**Sobre o parâmetro `n_init`:**

- Como o resultado depende da inicialização, cada execução pode cair em um mínimo local diferente e produzir clusters distintos.
- O `n_init` define quantas vezes o K-Means roda com sementes diferentes, ficando ao final com o melhor resultado entre todas as tentativas.
- Fixamos `n_init=10` para ter estabilidade; nas versões mais recentes do scikit-learn, o padrão passou a ser `'auto'`.

### 5.2. Escolhendo `k` — Cotovelo **e** Silhouette

**O Método do Cotovelo (Elbow):**

- A ideia é rodar o K-Means para vários valores de `k` e plotar a inércia de cada um em um gráfico.
- Procuramos o "cotovelo" da curva: o ponto a partir do qual a inércia para de cair de forma acentuada e passa a diminuir devagar, indicando que adicionar mais clusters já não compensa.
- A limitação é que essa leitura é visual e subjetiva — muitas vezes o cotovelo é ambíguo e duas pessoas leem valores diferentes no mesmo gráfico.

**O Silhouette Score como complemento quantitativo:**

- Para cada ponto, o silhouette mede o quão próximo ele está dos colegas do próprio cluster em comparação com o cluster vizinho mais próximo.
- O valor vai de -1 a 1, e quanto mais alto, melhor é a separação entre os grupos.
- Diferentemente do cotovelo, ele entrega um número objetivo, o que permite comparar candidatos a `k` sem depender de interpretação visual.

**Por que usar os dois juntos:** o cotovelo aponta uma faixa razoável de valores e o silhouette ajuda a escolher, dentro dessa faixa, o `k` que produz a melhor separação de fato.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

wcss = []
sil = []
range_k = range(2, 11)   # silhouette exige k >= 2

for k in range_k:
    km = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=42)
    labels = km.fit_predict(rfm_scaled)
    wcss.append(km.inertia_)
    sil.append(silhouette_score(rfm_scaled, labels))

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
axes[0].plot(list(range_k), wcss, marker='o', linestyle='--')
axes[0].set_title('Método do Cotovelo'); axes[0].set_xlabel('k'); axes[0].set_ylabel('WCSS (inércia)')
axes[0].grid(True)

axes[1].plot(list(range_k), sil, marker='o', color='darkorange')
axes[1].set_title('Silhouette Score por k'); axes[1].set_xlabel('k'); axes[1].set_ylabel('Silhouette')
axes[1].grid(True)
plt.tight_layout()
plt.show()

for k, s in zip(range_k, sil):
    print(f"k={k}: silhouette={s:.3f}")

> **Como decidir na prática:** procure o cotovelo no gráfico da esquerda e o pico (ou um platô alto) do silhouette no gráfico da direita. Para o Online Retail, `k=4` costuma ser um bom equilíbrio entre qualidade estatística dos clusters e facilidade de interpretação para o negócio. Ainda assim, deixe a escolha ser guiada pelos seus gráficos: se eles apontarem outro valor, use-o em vez de fixar um número por hábito.

### 5.3. Treinando o Modelo Final

**O que fazemos aqui:**

- Com o número de clusters definido, treinamos o modelo final que de fato será usado.
- Definimos um `random_state` para garantir que os resultados sejam reprodutíveis — rodando de novo, obtemos exatamente a mesma segmentação.
- Usamos `fit_predict`, que é um atalho conveniente para treinar o modelo e já obter o rótulo de cluster de cada cliente em uma única chamada.
- Guardamos esses rótulos na tabela RFM **original** (a não transformada), porque é sobre os valores em reais e dias que vamos interpretar os grupos mais adiante.

In [ ]:
K = 4  # ajuste conforme seus gráficos da Seção 5.2

kmeans = KMeans(n_clusters=K, init='k-means++', n_init=10, random_state=42)
clusters = kmeans.fit_predict(rfm_scaled)

rfm['Cluster'] = clusters
print("Tabela RFM com cluster atribuído:")
print(rfm.head())
print("\nTamanho de cada cluster:")
print(rfm['Cluster'].value_counts().sort_index())

### Exercício Prático 4: Experimentando com `k`

**Tarefa:** treine dois modelos K-Means, um com `k=3` e outro com `k=5`, e imprima a `.inertia_` de cada um para comparar.

**Perguntas para discutir em grupo:**

- Qual dos dois tem a menor inércia, e por que isso já era esperado antes mesmo de rodar?
- Se a inércia sempre diminui à medida que aumentamos `k`, por que não escolher simplesmente um `k` enorme para minimizá-la?
- Calcule também o **silhouette** dos dois modelos: o `k` de menor inércia é necessariamente o de melhor silhouette, ou os dois critérios podem discordar?

In [ ]:
# --- SEU CÓDIGO AQUI ---
# for k in [3, 5]:
#     km = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=42)
#     lab = km.fit_predict(rfm_scaled)
#     print(f"k={k}: inércia={km.inertia_:.1f} | silhouette={silhouette_score(rfm_scaled, lab):.3f}")

## Seção 6: Dando Sentido aos Dados — Análise e Interpretação dos Segmentos

### 6.1. Análise dos Centróides

**Por que os centróides são o coração da interpretação:**

- O K-Means devolve rótulos numéricos (0, 1, 2, 3) que estão matematicamente corretos, mas são comercialmente inúteis por si sós — ninguém sabe o que "cluster 2" significa.
- O valor real da segmentação nasce da interpretação: precisamos descobrir o que cada grupo representa em termos de comportamento de compra.
- Cada centróide é, essencialmente, o "cliente médio" do seu cluster, e por isso funciona como a assinatura RFM daquele grupo.

**Trazendo os centróides de volta para a escala original:**

- Os centróides que o modelo fornece estão na escala transformada (log seguido de padronização), então seus números não têm significado direto.
- Para lê-los em dias, número de compras e libras, revertemos os dois passos na ordem inversa: primeiro `scaler.inverse_transform(...)` desfaz a padronização e devolve a escala logarítmica, e depois `np.expm1(...)` desfaz o `log1p` e recupera a escala original de R, F e M.

In [ ]:
centroids_scaled = kmeans.cluster_centers_

# Desfaz padronização -> escala log; depois desfaz o log -> escala original
centroids_log = scaler.inverse_transform(centroids_scaled)
centroids_original = np.expm1(centroids_log)

centroids_df = pd.DataFrame(centroids_original, columns=['Recency', 'Frequency', 'Monetary'])
centroids_df.index.name = 'Cluster'
print("Centróides na escala ORIGINAL (R em dias, F em nº de compras, M em £):")
print(centroids_df)

### 6.2. Nomeando os Clusters de Forma **Robusta**

**O problema do mapa fixo (o bug que estamos corrigindo):**

- O notebook original usava um dicionário fixo, do tipo `{0: 'Campeões', 1: 'Adormecidos', ...}`, para dar nome a cada cluster.
- Isso quebra na prática, porque os rótulos numéricos do K-Means são arbitrários e mudam conforme o `random_state`, a versão do scikit-learn ou os próprios dados.
- Na maioria das execuções, o cluster que recebe o número 0 não é o mesmo de antes, então as personas acabam trocadas — os "Campeões" viram "Adormecidos" sem ninguém perceber.

**A solução — deixar o comportamento decidir o nome:**

- Construímos um pequeno **RFM score** a partir dos centróides, premiando recência baixa, frequência alta e valor monetário alto.
- Ordenamos os clusters por esse score e atribuímos as personas segundo o ranking, e não segundo o número do rótulo.
- Assim, o cluster de melhor comportamento sempre recebe "Campeões", qualquer que seja o número que o K-Means tenha dado a ele naquela execução.

In [ ]:
# Score de qualidade do cliente a partir dos centróides (quanto maior, melhor)
rank = centroids_df.copy()
rank['R_rank'] = rank['Recency'].rank(ascending=True)    # recência baixa = melhor
rank['F_rank'] = rank['Frequency'].rank(ascending=False) # frequência alta = melhor
rank['M_rank'] = rank['Monetary'].rank(ascending=False)  # monetary alto = melhor
rank['Score'] = rank[['R_rank', 'F_rank', 'M_rank']].mean(axis=1)

# Ordena do melhor (menor Score) para o pior
ordem = rank['Score'].sort_values().index.tolist()

# Personas por posição no ranking (para k=4). Ajuste a lista se usar outro k.
personas = ['Clientes Campeões',
            'Clientes Leais / Potenciais',
            'Clientes em Risco',
            'Clientes Adormecidos']

# Se k != 4, gera nomes genéricos para não quebrar
if len(ordem) != len(personas):
    personas = [f'Segmento {i+1}' for i in range(len(ordem))]

cluster_names = {cluster_id: personas[pos] for pos, cluster_id in enumerate(ordem)}
print("Mapeamento robusto rótulo -> persona:")
for cid in sorted(cluster_names):
    print(f"  Cluster {cid}: {cluster_names[cid]}")

rfm['Segment'] = rfm['Cluster'].map(cluster_names)

### 6.3. Perfil de Negócio de Cada Segmento

**Fechando o ciclo com o Princípio de Pareto:**

- Dar nome aos grupos é só parte do trabalho; o negócio também quer saber o **tamanho** de cada segmento e o **peso** dele na receita.
- Para cada segmento, calculamos as médias de R, F e M, o número de clientes e o percentual que ele representa da base, além da receita total e do percentual que ele representa da receita.
- É aqui que a regra 80/20 deixa de ser teoria: conseguimos ver, em números, se um grupo pequeno de "Campeões" concentra uma fatia desproporcional da receita, o que orienta diretamente onde o marketing deve concentrar esforços.

In [ ]:
perfil = rfm.groupby('Segment').agg(
    Recency=('Recency', 'mean'),
    Frequency=('Frequency', 'mean'),
    Monetary=('Monetary', 'mean'),
    N_Clientes=('Monetary', 'size'),
    Receita_Total=('Monetary', 'sum'),
)
perfil['%_Base'] = (100 * perfil['N_Clientes'] / perfil['N_Clientes'].sum())
perfil['%_Receita'] = (100 * perfil['Receita_Total'] / perfil['Receita_Total'].sum())
perfil = perfil.sort_values('%_Receita', ascending=False)

print("Perfil de negócio por segmento:")
print(perfil.round(2))

### 6.4. Visualizando os Segmentos — Snake Plot

**O que é o Snake Plot e por que ele comunica bem:**

- É um gráfico de linhas em que cada linha representa um segmento; o eixo X mostra as três métricas RFM e o eixo Y mostra o valor médio padronizado (o centróide) de cada uma.
- O formato de cada "cobra" revela de relance o perfil do grupo — uma linha alta em Frequência e Monetary e baixa em Recência grita "Campeões", por exemplo.
- É uma das ferramentas mais eficazes para comunicar segmentação a stakeholders não técnicos, como a equipe de marketing.

> Construímos o gráfico a partir do **mapeamento robusto** definido na Seção 6.2, de modo que os nomes exibidos correspondam de fato ao comportamento de cada cluster, sem depender da ordem dos rótulos.

In [ ]:
# Centróides na escala padronizada, já com o nome correto do segmento
centroids_scaled_df = pd.DataFrame(centroids_scaled, columns=['Recency', 'Frequency', 'Monetary'])
centroids_scaled_df['Segment'] = [cluster_names[i] for i in range(len(centroids_scaled_df))]

datamart_melt = pd.melt(centroids_scaled_df, id_vars='Segment',
                        value_vars=['Recency', 'Frequency', 'Monetary'],
                        var_name='Metric', value_name='Value')

plt.figure(figsize=(12, 7))
sns.lineplot(data=datamart_melt, x='Metric', y='Value', hue='Segment', palette='viridis', marker='o')
plt.axhline(0, color='black', linestyle='--')   # média da população
plt.title('Snake Plot dos Segmentos de Clientes')
plt.xlabel('Métricas RFM'); plt.ylabel('Valor Padronizado (Centróide)')
plt.legend(title='Segmento')
plt.grid(True)
plt.show()

### 6.5. Visualizando os Clientes em 2D

**Por que acrescentar um gráfico de dispersão:**

- O snake plot mostra apenas os centróides, ou seja, o resumo de cada grupo; aqui queremos ver os **clientes de verdade** espalhados e coloridos por segmento.
- Um gráfico de Frequência contra Monetary, ambos em escala logarítmica, torna os grupos tangíveis e mostra como eles ocupam regiões diferentes do espaço.
- Ele também ajuda a enxergar sobreposições entre segmentos, revelando os clientes de fronteira que poderiam ter caído em mais de um grupo.

In [ ]:
plt.figure(figsize=(10, 7))
sns.scatterplot(data=rfm, x='Frequency', y='Monetary', hue='Segment',
                palette='viridis', alpha=0.6, s=30)
plt.xscale('log'); plt.yscale('log')
plt.title('Clientes por Segmento (Frequência × Monetary, escala log)')
plt.xlabel('Frequência (log)'); plt.ylabel('Monetary (log)')
plt.legend(title='Segmento')
plt.show()

## Seção 7: Uma Alternativa Clássica — RFM por Quantis (sem ML)

Antes de encerrar, vale conhecer a versão **tradicional do marketing** para o RFM, que não usa nenhum algoritmo de machine learning. Comparar essa abordagem com o K-Means é muito instrutivo, porque mostra que clustering é uma escolha entre várias, e não a única maneira de segmentar.

### 7.1. A Ideia do Scoring por Quantis

**Como o método funciona:**

- Em vez de agrupar por distância, dividimos cada métrica em faixas (quantis) e damos a cada cliente uma nota de 1 a 5 conforme a faixa em que ele cai.
- A função `qcut` do pandas faz esse corte, separando a distribuição em partes de tamanho aproximadamente igual.
- É preciso cuidado com a **direção** de cada nota: para a Recência, quanto menor melhor, então os clientes mais recentes recebem nota 5; já para Frequência e Monetary, quanto maior melhor, então quem mais compra e mais gasta é que recebe nota 5.

**Como combinar as notas em um score:**

- Uma forma é concatenar as três notas, gerando códigos como "555" (o cliente ideal, topo em tudo) ou "111" (o pior caso).
- Outra é somar as notas, produzindo um score único que vai de 3 (mínimo) a 15 (máximo), mais fácil de ordenar e de usar em regras.

In [ ]:
# Notas de 1 a 5 por quantil
rfm_q = rfm[['Recency', 'Frequency', 'Monetary']].copy()

rfm_q['R_score'] = pd.qcut(rfm_q['Recency'], q=5, labels=[5, 4, 3, 2, 1]).astype(int)
# 'rank(method="first")' evita erro do qcut quando há muitos valores repetidos
rfm_q['F_score'] = pd.qcut(rfm_q['Frequency'].rank(method='first'), q=5, labels=[1, 2, 3, 4, 5]).astype(int)
rfm_q['M_score'] = pd.qcut(rfm_q['Monetary'], q=5, labels=[1, 2, 3, 4, 5]).astype(int)

rfm_q['RFM_Score'] = rfm_q[['R_score', 'F_score', 'M_score']].sum(axis=1)
rfm_q['RFM_Segment'] = (rfm_q['R_score'].astype(str)
                        + rfm_q['F_score'].astype(str)
                        + rfm_q['M_score'].astype(str))

print(rfm_q.head())
print("\nDistribuição do RFM_Score (3 = pior, 15 = melhor):")
print(rfm_q['RFM_Score'].value_counts().sort_index())

### 7.2. Rotulando Segmentos com Regras Simples

- Com as notas calculadas, dá para nomear segmentos usando regras de negócio legíveis, que qualquer pessoa da equipe consegue entender e auditar.
- O exemplo abaixo mapeia faixas do score somado para personas, mas os limites são uma decisão estratégica: ajuste-os conforme a política de relacionamento da empresa e o tamanho de grupo que você quer acionar em cada campanha.

In [ ]:
def rotular(score):
    if score >= 13:
        return 'Campeões'
    elif score >= 10:
        return 'Leais'
    elif score >= 7:
        return 'Potenciais / Em risco'
    else:
        return 'Adormecidos'

rfm_q['Label'] = rfm_q['RFM_Score'].apply(rotular)
print(rfm_q['Label'].value_counts())

### 7.3. K-Means vs. Quantis: Quando Usar Cada Um?

**A favor do RFM por quantis (baseado em regras):**

- É simples, transparente e muito fácil de explicar ao marketing, já que cada cliente cai em uma faixa por um critério explícito.
- Não exige normalização, não exige escolher `k` e não depende de aleatoriedade, o que o torna rápido de colocar de pé.
- Em contrapartida, suas fronteiras são fixas e um tanto arbitrárias, e ele trata as três métricas de forma independente, sem enxergar como elas se combinam.

**A favor do K-Means (clustering):**

- Ele **descobre** grupos naturais a partir da estrutura conjunta de R, F e M, em vez de impor cortes definidos por nós de antemão.
- Adapta-se aos dados e pode revelar segmentos não óbvios que um esquema de faixas jamais encontraria.
- Por outro lado, é menos interpretável (nem sempre é claro por que um cliente caiu em certo grupo) e exige um pré-processamento cuidadoso, com log, escala e escolha de `k`.

> **A mensagem para levar:** clustering é uma forma de fazer RFM, não a única. Muitas empresas começam pelo scoring por quantis, por ser rápido de comunicar, e evoluem para clustering quando querem descobrir segmentos mais sutis que as regras fixas não capturam.

## Seção 8: Um Segundo Olhar — Clustering Hierárquico

Para entender melhor tanto os acertos quanto as limitações do K-Means, vamos comparar seus resultados com os do clustering **hierárquico aglomerativo**, um método com uma lógica bem diferente de formar grupos.

### 8.1. Como Funciona o Hierárquico Aglomerativo

**A lógica de baixo para cima (bottom-up):**

- O método começa tratando **cada cliente como o seu próprio cluster**, ou seja, com tantos clusters quantos forem os pontos.
- A cada passo, ele encontra os dois clusters mais próximos e os funde em um só, reduzindo o número total de grupos aos poucos.
- Esse processo se repete até que sobre um único cluster contendo todo mundo, e o histórico de fusões forma uma árvore hierárquica.

**As diferenças em relação ao K-Means:**

- Ao contrário do K-Means, não é preciso definir `k` de antemão; o número de clusters é escolhido depois, "cortando" a árvore na altura que fizer mais sentido.
- O método produz um **dendrograma**, um gráfico que mostra toda a estrutura de fusões e ajuda a sugerir quantos grupos realmente existem.
- Em compensação, o custo computacional cresce rapidamente com o número de pontos, então usamos uma **amostra** dos dados para desenhar o dendrograma em datasets grandes.

### 8.2. O Dendrograma

**Como ler o gráfico:**

- No eixo X ficam os clientes (ou grupos deles já fundidos), e no eixo Y fica a **distância** em que cada fusão aconteceu.
- Se traçarmos uma linha horizontal em certa altura, o número de linhas verticais que ela cruza indica em quantos clusters os dados ficam divididos naquele nível.
- Fusões que ocorrem a uma altura muito maior do que as anteriores sinalizam grupos bem separados, e o "salto" logo abaixo dessas fusões costuma ser um bom lugar para cortar a árvore.

In [ ]:
from scipy.cluster.hierarchy import dendrogram, linkage
from sklearn.cluster import AgglomerativeClustering

# Amostra para o dendrograma (hierárquico é O(n^2) — inviável plotar milhares de pontos)
rng = np.random.RandomState(42)
n_amostra = min(1000, rfm_scaled.shape[0])
idx = rng.choice(rfm_scaled.shape[0], size=n_amostra, replace=False)
amostra = rfm_scaled[idx]

# Método de Ward: minimiza a variância intra-cluster (combina bem com dados escalados)
Z = linkage(amostra, method='ward')

plt.figure(figsize=(12, 5))
dendrogram(Z, truncate_mode='lastp', p=20, show_leaf_counts=True)
plt.title('Dendrograma (amostra, método de Ward)')
plt.xlabel('Clientes agrupados'); plt.ylabel('Distância (Ward)')
plt.axhline(y=Z[-4, 2], color='red', linestyle='--', label='corte sugerido (~4 clusters)')
plt.legend()
plt.show()

### 8.3. Comparando os Rótulos: K-Means × Hierárquico

**O que fazemos nesta comparação:**

- Ajustamos o `AgglomerativeClustering` usando o **mesmo `k`** do K-Means, sobre exatamente os mesmos dados já transformados e escalados.
- Cruzamos os rótulos dos dois métodos em uma tabela de contingência, para ver quanto eles concordam entre si.

**Como interpretar o resultado:**

- Se os métodos concordam bastante, cada cluster de um casa fortemente com um cluster do outro, o que aparece como uma célula bem grande em cada linha e coluna da tabela.
- As divergências apontam os clientes de fronteira, aqueles cuja atribuição depende do algoritmo escolhido.
- Vale lembrar que os **números** dos clusters não coincidem entre os métodos: o que importa não é se o "0" de um é o "0" do outro, mas se o padrão geral de agrupamento é parecido — e o Adjusted Rand Index resume isso em um único número.

In [ ]:
agg = AgglomerativeClustering(n_clusters=K, linkage='ward')
labels_agg = agg.fit_predict(rfm_scaled)

comparacao = pd.crosstab(rfm['Cluster'], labels_agg,
                         rownames=['K-Means'], colnames=['Hierárquico'])
print("Tabela de contingência (K-Means × Hierárquico):")
print(comparacao)

from sklearn.metrics import adjusted_rand_score
ari = adjusted_rand_score(rfm['Cluster'], labels_agg)
print(f"\nAdjusted Rand Index (concordância entre os métodos): {ari:.3f}")
print("(1.0 = concordância total; ~0 = concordância ao acaso)")

## Seção 9: Conclusão, Produção e Próximos Passos

### 9.1. Resumo do Projeto

Percorremos, do início ao fim, um projeto completo de ciência de dados aplicado a um problema real de negócio:

- Fizemos uma **limpeza** guiada por regras de negócio, definindo com clareza o que conta como cliente analisável.
- Aplicamos **engenharia de features** para transformar transações soltas em perfis de cliente resumidos por R, F e M.
- Cuidamos do **pré-processamento** de forma correta, usando `log1p` para domar a assimetria e `StandardScaler` para igualar as escalas.
- Fizemos a **modelagem** com K-Means, escolhendo `k` com o apoio conjunto do cotovelo e do silhouette.
- Chegamos a uma **interpretação robusta**, com personas derivadas dos centróides e um perfil de negócio que cruza percentual da base com percentual da receita.
- E ainda **comparamos abordagens**, colocando o K-Means lado a lado com o RFM por quantis e com o clustering hierárquico.

O resultado final não são números soltos, mas personas acionáveis, prontas para orientar campanhas de marketing personalizadas.

### 9.2. Aplicações no Mundo Real para Desenvolvedores Web

- **Integração com sistemas de CRM:** os segmentos gerados podem ser exportados e usados para "etiquetar" (tag) os clientes no CRM, alimentando fluxos de automação de marketing.
- **Personalização de conteúdo em tempo real:** um sistema web pode consultar o segmento do cliente logado e adaptar dinamicamente a experiência — banners, cupons e recomendações mudam conforme a pessoa seja um "Campeão" ou um cliente "Em risco".
- **Criação de uma API de segmentação:** o modelo K-Means e o `StandardScaler`, uma vez serializados, podem ser encapsulados em um microsserviço que recebe os valores de R, F e M de um cliente e devolve o segmento correspondente para outros sistemas.

> **Atenção ao pipeline em produção:** o serviço precisa aplicar exatamente as mesmas transformações do treino e na mesma ordem — `log1p`, depois `scaler.transform` e só então `kmeans.predict`. É justamente por isso que salvamos o `scaler` junto com o modelo, para que o pré-processamento em produção seja idêntico ao do treinamento.

### 9.3. Serializando o Pipeline (scaler + modelo)

- Salvamos o scaler e o modelo juntos, em um único arquivo, usando `joblib`, para que nunca sejam usados de forma desencontrada.
- Em seguida, demonstramos como classificar um cliente novo reproduzindo o pipeline completo passo a passo, exatamente como um serviço de produção faria a cada requisição.

In [ ]:
import joblib

# Salva scaler e modelo juntos (a ordem de uso é log1p -> scaler -> kmeans)
artefato = {'scaler': scaler, 'kmeans': kmeans, 'cluster_names': cluster_names}
joblib.dump(artefato, 'modelo_segmentacao_rfm.joblib')
print("Artefato salvo em modelo_segmentacao_rfm.joblib")

# --- Inferência para um cliente novo ---
art = joblib.load('modelo_segmentacao_rfm.joblib')

cliente_novo = pd.DataFrame([{'Recency': 12, 'Frequency': 8, 'Monetary': 3500.0}])
X_novo = np.log1p(cliente_novo[['Recency', 'Frequency', 'Monetary']])   # 1) log
X_novo = art['scaler'].transform(X_novo)                                # 2) escala
cluster_novo = art['kmeans'].predict(X_novo)[0]                         # 3) prediz
print(f"\nCliente novo -> Cluster {cluster_novo} -> {art['cluster_names'][cluster_novo]}")

## Seção 10: Desafio Final — Segmentando Clientes no E-commerce Brasileiro (Olist)

### 10.1. O Novo Dataset

- O desafio usa o **Brazilian E-Commerce Public Dataset by Olist**, disponível no Kaggle, com informações de cerca de 100.000 pedidos realizados no Brasil.
- A grande diferença em relação ao Online Retail é que aqui a informação está espalhada por **várias tabelas** que precisam ser unidas antes de qualquer análise.
- Isso torna o desafio mais próximo de um cenário real, em que os dados quase nunca chegam prontos em um único arquivo.

### 10.2. ⚠️ Duas Armadilhas Específicas do Olist

Antes de começar a codificar, preste atenção a dois pontos que costumam gerar erros silenciosos e resultados sutilmente errados:

- **A identidade do cliente:** use sempre o **`customer_unique_id`**, e não o `customer_id`. Na Olist, o `customer_id` é gerado por pedido, de modo que a mesma pessoa recebe um identificador diferente a cada compra — se você usá-lo, todo cliente vai parecer ter feito exatamente uma compra e a métrica de Frequência perderá completamente o sentido.
- **O grão da tabela de itens:** a `order_items` tem uma linha por item, não por pedido. Somar `price` depois de unir tudo está correto para o Monetary, porque queremos justamente o total gasto; mas para a Frequência é preciso contar `order_id` **únicos** com `nunique`, senão você acaba contando itens em vez de pedidos. Vale também decidir conscientemente se o `freight_value` (o frete) entra ou não no Monetary, já que ele faz parte do que o cliente efetivamente pagou.

### 10.3. O Roteiro do Desafio

1. **Carregar** os arquivos `olist_customers_dataset.csv`, `olist_orders_dataset.csv` e `olist_order_items_dataset.csv`.
2. **Unir** as tabelas na sequência pedidos → clientes → itens, de modo que cada linha final tenha cliente, data e preço.
3. **Limpar** os dados, mantendo apenas os pedidos efetivamente entregues (`order_status == 'delivered'`) e convertendo as colunas de data.
4. **Construir o RFM** agrupando por `customer_unique_id`: Recência a partir de `order_purchase_timestamp`, Frequência pela contagem de `order_id` únicos e Monetary pela soma de `price` (decidindo o que fazer com o frete).
5. **Pré-processar** exatamente como na aula, com `log1p` seguido de `StandardScaler` — a mesma correção que fez toda a diferença aqui.
6. **Modelar** com o apoio do cotovelo e do silhouette e então treinar o K-Means.
7. **Interpretar** os centróides, criar personas de forma robusta, montar o perfil de negócio e desenhar o snake plot.
8. **Bônus:** comparar os perfis de clientes do e-commerce brasileiro com os do varejista do Reino Unido, discutindo semelhanças e diferenças culturais de consumo.

In [ ]:
print("Iniciando o Desafio Final — Dataset Olist\n")

try:
    customers   = pd.read_csv('olist_customers_dataset.csv')
    orders      = pd.read_csv('olist_orders_dataset.csv')
    order_items = pd.read_csv('olist_order_items_dataset.csv')

    # Une pedidos -> clientes (traz customer_unique_id) -> itens (traz price)
    df_olist = orders.merge(customers, on='customer_id', how='inner')
    df_olist = df_olist.merge(order_items, on='order_id', how='inner')

    # Mantém apenas pedidos entregues e converte a data
    df_olist = df_olist[df_olist['order_status'] == 'delivered'].copy()
    df_olist['order_purchase_timestamp'] = pd.to_datetime(df_olist['order_purchase_timestamp'])

    print("Olist unido. Formato:", df_olist.shape)
    print(df_olist[['customer_unique_id', 'order_id', 'order_purchase_timestamp', 'price']].head())

    # ---- ESQUELETO DO RFM (complete como exercício) ----
    # snapshot = df_olist['order_purchase_timestamp'].max() + dt.timedelta(days=1)
    # rfm_olist = df_olist.groupby('customer_unique_id').agg(
    #     Recency=('order_purchase_timestamp', lambda d: (snapshot - d.max()).days),
    #     Frequency=('order_id', 'nunique'),      # <- order_id ÚNICOS, não linhas
    #     Monetary=('price', 'sum'),
    # )
    # A partir daqui: log1p -> StandardScaler -> cotovelo/silhouette -> K-Means -> interpretação

except FileNotFoundError:
    print("Arquivos do Olist não encontrados. Baixe do Kaggle para executar o desafio.")
    print("Link: https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce")

### 10.4. Reflexão de Encerramento

Ao longo desta prática, você aplicou de ponta a ponta o fluxo de um projeto de segmentação e vestiu, na sequência, três papéis:

- Como **arquiteto de dados**, uniu múltiplas tabelas respeitando o grão correto de cada uma.
- Como **engenheiro de dados**, limpou os dados, tratou a assimetria e a escala e evitou as armadilhas de grão e de identidade que derrubam muita análise.
- Como **cientista de dados**, escolheu `k` com critério, comparou algoritmos diferentes e traduziu clusters abstratos em personas de negócio.

O mesmo esqueleto que praticamos aqui — limpeza, cálculo do RFM, log e escala, K-Means com cotovelo e silhouette, e finalmente interpretação robusta — serve para praticamente qualquer problema de segmentação de clientes que você venha a encontrar na carreira.